# 03 - MMM Baseline Model

Bayesian Marketing Mix Model using PyMC-Marketing.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from pymc_marketing.mmm import (
    MMM,
    GeometricAdstock,
    LogisticSaturation,
)

SEED = 42
rng = np.random.default_rng(SEED)
az.style.use("arviz-darkgrid")
print("OK")

## 2. Load and Prepare Data

In [ ]:
df = pd.read_parquet("../data/processed/mmm_data.parquet")
print(f"Raw shape: {df.shape}")

df["date"] = pd.to_datetime(df["DATE_DAY"])

TOP_REGION = df.groupby("TERRITORY_NAME")["ALL_PURCHASES_ORIGINAL_PRICE"].sum().idxmax()
print(f"Top region: {TOP_REGION}")

df_region = df[df["TERRITORY_NAME"] == TOP_REGION].copy()

In [ ]:
df_region["week"] = df_region["date"].dt.to_period("W").dt.start_time

SPEND_COLS = [
    "GOOGLE_PAID_SEARCH_SPEND", "GOOGLE_SHOPPING_SPEND",
    "META_FACEBOOK_SPEND", "META_INSTAGRAM_SPEND", "TIKTOK_SPEND"
]
TARGET_COL = "ALL_PURCHASES_ORIGINAL_PRICE"

agg_cols = [c for c in SPEND_COLS + [TARGET_COL] if c in df_region.columns]

df_weekly = df_region.groupby("week")[agg_cols].sum().reset_index()
df_weekly = df_weekly.sort_values("week").reset_index(drop=True)

print(f"Weekly rows: {len(df_weekly)}")
print(f"Unique weeks: {df_weekly['week'].nunique()}")

assert df_weekly["week"].is_unique, "ERROR: Dates not unique!"
print("Dates are unique - OK")

In [ ]:
DATE_COL = "week"
available_channels = [c for c in SPEND_COLS if c in df_weekly.columns]

X = df_weekly[[DATE_COL] + available_channels].copy()
X = X.fillna(0)

y = df_weekly[TARGET_COL].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Channels: {available_channels}")

## 3. Model

In [ ]:
mmm = MMM(
    date_column=DATE_COL,
    channel_columns=available_channels,
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    yearly_seasonality=2,
)
print("Model defined")

## 4. Fit

In [ ]:
%%time
mmm.fit(
    X=X,
    y=y,
    chains=2,
    draws=500,
    target_accept=0.9,
    random_seed=rng,
)
print("Fit complete")

## 5. Diagnostics

In [ ]:
summary = az.summary(mmm.idata, var_names=["~lam"], filter_vars="like")
bad_rhat = summary[summary["r_hat"] > 1.05]
print(f"Params with r_hat > 1.05: {len(bad_rhat)}")
summary.head(10)

In [ ]:
az.plot_trace(mmm.idata, var_names=["intercept"], figsize=(10, 4))
plt.tight_layout()
plt.show()

## 6. Results

In [ ]:
# Sample posterior predictive
pp = mmm.sample_posterior_predictive(X, extend_idata=True)

# Get predictions from idata
y_pred = mmm.idata.posterior_predictive["y"].mean(dim=["chain", "draw"]).values

mae = np.abs(y - y_pred).mean()
r2 = 1 - ((y - y_pred)**2).sum() / ((y - y.mean())**2).sum()

print(f"MAE: {mae:,.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y, label="Actual", alpha=0.7)
ax.plot(y_pred, label="Predicted", alpha=0.7)
ax.set_title(f"Model Fit - {TOP_REGION}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
contributions = mmm.compute_channel_contribution_original_scale()
mean_contrib = contributions.mean(dim=["chain", "draw"])
total = mean_contrib.sum(dim="date").to_dataframe(name="contribution")

fig, ax = plt.subplots(figsize=(10, 5))
total.sort_values("contribution").plot(kind="barh", ax=ax, legend=False, color="steelblue")
ax.set_xlabel("Total Contribution")
ax.set_title("Channel Contributions")
plt.tight_layout()
plt.show()

In [ ]:
roi_data = []
for ch in available_channels:
    spend = X[ch].sum()
    contrib = total.loc[ch, "contribution"] if ch in total.index else 0
    roi = contrib / spend if spend > 0 else 0
    roi_data.append({"channel": ch.replace("_SPEND", ""), "spend": spend, "roi": roi})

roi_df = pd.DataFrame(roi_data).sort_values("roi", ascending=False)
roi_df

## 7. Save

In [ ]:
from pathlib import Path
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)
mmm.idata.to_netcdf(MODEL_DIR / "mmm_baseline_trace.nc")
roi_df.to_csv(MODEL_DIR / "roi_baseline.csv", index=False)
print("Saved.")